# 04 · COLMAP Multi-View Reconstruction  ← **Milestone Artifact #2**

**Goal:** produce a sparse 3D point cloud of one tree showing primary
branch structure, using COLMAP for full multi-view SfM. This is the
second milestone artifact: a cloud you can point at and say "this is the
tree."

Library-wrapped — we shell out to the `colmap` CLI. The from-scratch
two-view alternative lives in notebook 03; the two-view-vs-multi-view
ablation in notebook 08 compares them quantitatively.

### Why COLMAP for the milestone

COLMAP gives us pose for ALL ~60 frames at once with proper bundle
adjustment. A from-scratch multi-view BA is out of scope for CS 131 (it's
a research-grade implementation), but the two-view from-scratch pipeline
in notebook 03 hits the algorithmic content the assignment cares about.

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import sfm, viz

    TREE_ID = "tree_oak_01"
    FRAMES_DIR = PROJECT_ROOT / "data" / "frames" / TREE_ID
    WORKSPACE = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_colmap"


if not FRAMES_DIR.exists():
    raise FileNotFoundError(
        f"Frames directory not found at {FRAMES_DIR}.\n"
        "Run notebook 01 to extract frames for this tree."
    )

## 1. Run COLMAP automatic_reconstructor

This takes a few minutes on a laptop CPU; if you have a CUDA build it's
much faster. Skip the cell on re-runs if `<workspace>/sparse/0/` already
contains a reconstruction.

In [ ]:
sparse_dir = WORKSPACE / "sparse" / "0"
if not sparse_dir.exists():
    sparse_dir = sfm.run_colmap(
        image_dir=FRAMES_DIR, workspace_dir=WORKSPACE,
        quality="medium", single_camera=True, use_gpu=False,
    )
    print(f"COLMAP produced model at {sparse_dir}")
else:
    print(f"Reusing existing COLMAP model at {sparse_dir}")

## 2. Load the sparse model + save the cloud as PLY

In [ ]:
recon = sfm.load_sparse_model(sparse_dir)
print(f"Registered images: {len(recon.image_names)}")
print(f"3D points:         {len(recon.points_3d)}")

ply_path = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_sparse.ply"
sfm.save_ply(ply_path, recon.points_3d, recon.point_colors)
print(f"Saved → {ply_path}")

## 3. Render the milestone figure

Save a clean 3D scatter for the writeup. This is the cloud-of-one-tree
promised in the milestone.

In [ ]:
fig = viz.plot_point_cloud(
    recon.points_3d, recon.point_colors,
    title=f"{TREE_ID} — sparse multi-view cloud ({len(recon.points_3d)} pts)",
)
viz.save_fig(fig, "milestone_cloud.png")
plt.show()

## 4. Save projection matrices for downstream stages

Notebook 06 (reprojection filter) needs per-image P = K [R | t].

In [ ]:
Ps = sfm.projection_matrices(recon)
np.savez(
    PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_poses.npz",
    projections=np.array(Ps),
    image_names=np.array(recon.image_names),
)
print(f"Saved {len(Ps)} projection matrices.")